# Calculate Depression Depths per Subbasin

## Contents
- [Input Datasets for this Notebook](#input-datasets-for-this-notebook)
- [0. Import required modules](#0-import-required-modules)
- [1. Organize data folders and files](#1-organize-data-folders-and-files)
    - [1.1 Create "DepressionDepths" subfolder](#11-create-depressiondepths-subfolder)
    - [1.2 Create subfolders in "DepressionDepths" subfolder ("Raw", "Interim", "Processed")](#12-create-subfolders-in-depressiondepths-subfolder-raw-interim-processed)
    - [1.3 Move input data files to "Raw" subfolder](#13-move-input-data-files-to-raw-subfolder)
- [2. Declare input data file names as variables](#2-declare-input-data-file-names-as-variables)
- [3. Process LiDAR raster DEM](#3-process-lidar-raster-dem)
    - [3.1 Project subbasins shapefile to align with LiDAR raster DEM](#31-project-subbasins-shapefile-to-align-with-lidar-raster-dem)
    - [3.2 Clip LiDAR raster DEM to watershed](#32-clip-lidar-raster-dem-to-watershed)
    - [3.3 Resample clipped LiDAR raster DEM to 5m resolution](#33-resample-clipped-lidar-raster-dem-to-5m-resolution)
- [4. Process NHN Streams Network Shapefile](#4-process-nhn-streams-network-shapefile)
    - [4.1 Project subbasins shapefile to align with NHN streams shapefile](#41-project-subbasins-shapefile-to-align-with-nhn-streams-shapefile)
    - [4.2 Clip NHN streams shapefile to watershed](#42-clip-nhn-streams-shapefile-to-watershed)
    - [4.3 Visualize clipped and projected NHN streams shapefile](#43-visualize-clipped-and-projected-nhn-streams-shapefile)
- [5. Generate depressions raster](#5-generate-depressions-raster)
- [6. Calculate Depression Depths](#6-calculate-depression-depths)
    - [6.1 Calculate Subbasin Zonal Statistics](#61-calculate-subbasin-zonal-statistics)
    - [6.2 Calculate depression depths from zonal statistics](#62-calculate-depression-depths-from-zonal-statistics)
    - [6.3 Write depression depths to file](#63-write-depression-depths-to-file)
- [7. Visualize Processed Data](#7-visualize-processed-data)








## Input Datasets for this Notebook

Prior to running this notebook, the following zip files must be downloaded and extracted into a new project folder called "Data" at the root project directory:
- Watershed of interest based on outlet point from the [CLRH Hydrofabrics](https://hydrology.uwaterloo.ca/CLRH/Hydrofabric.html) 
    - e.g., ```ID: 05OE006``` for Manning Canal
- Streams dataset of interest from [NHN streams](https://ftp.maps.canada.ca/pub/nrcan_rncan/vector/geobase_nhn_rhn/shp_en/)
    - e.g., ```ngn_rhn_05oe000_shp_en.zip``` for Manning Canal
- Raster DEM of interest from [LiDAR DEMs](https://mli.gov.mb.ca/dems/index_external_lidar.html)
    - e.g., ```Seine & Rat 2016``` for Manning Canal

## 0. Import required modules

In [ ]:
# Sytem
import os
import sys
import fileinput
from pathlib import Path
import shutil


# Analysis
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, LineString, Polygon, MultiPoint, mapping
from shapely.geometry import shape as shapely_shape
import rasterio
from rasterio.crs import CRS
from rasterio.mask import mask
from bs4 import BeautifulSoup


# Visualization
import matplotlib.pyplot as plt
import rasterio.plot


# Whitebox
from WBT.whitebox_tools import WhiteboxTools


sys.path.append("..")
from src.py.proj_conf import *

## 1. Organize data folders and files

In [ ]:
class usr_input:
    def string(self, description, default_value=None):
        """
        Prompt the user for a string input with a description.
        If no input is provided, return the default value if specified.
        """
        prompt = f"Enter {description} (default: {default_value}): "
        user_input = input(prompt).strip()
        if user_input == "" and default_value is not None:
            print(f"No {description} provided. Default applied ('{default_value}').")
            return default_value
        elif user_input == "":
            while user_input == "":
                user_input = input(f"A value is required. " + prompt).strip()
        return user_input
    
    def file(self, description, default_value=None):
        """
        Prompt the user for a file name input with a description.
        If no input is provided, return the default value if specified.
        """
        prompt = f"Enter {description} filename (e.g., '{default_value}'): "
        filename = input(prompt).strip()
        if "." in filename:
            filename = input("Please do not include file extension in the name. " + prompt).strip()
        elif default_value is None and filename == "":
            while filename == "":
                filename = input(f"A file name is required. " + prompt).strip()
        elif filename == "":
            filename = default_value
            print(f"No {description} name provided. Default applied ('{default_value}').")
        return filename

# Usage:
# CLRH_FILENAME = usr_input().file()

### 1.1 Create "DepressionDepths" subfolder

In [ ]:
# Set watershed name
WATERSHED_NAME = usr_input().string("watershed", "Manning")


# Path for this notebook
NOTEBOOK_PATH = Path.cwd()


# Path for PWA root
PWA_PATH = str(NOTEBOOK_PATH.parent.as_posix())


# Path for base raw data folder
BS_DATA_PATH = PWA_PATH + r"/Data/"


# Specify path for watershed folder
WATERSHED_PATH = BS_DATA_PATH + WATERSHED_NAME


# Path for DepressionDepths folder
DEPRESSION_DEPTHS_PATH = WATERSHED_PATH + r"/DepressionDepths"


# Create watershed folder in specified path
os.makedirs(DEPRESSION_DEPTHS_PATH, 
            exist_ok=True) # Do nothing if already exists

### 1.2 Create subfolders in "DepressionDepths" subfolder ("Raw", "Interim", "Processed")

In [ ]:
# Specify paths for depression depths subfolders
DD_RAW_PATH = DEPRESSION_DEPTHS_PATH + r"/Raw/"
DD_INTERIM_PATH = DEPRESSION_DEPTHS_PATH + r"/Interim/"
DD_PROCESSED_PATH = DEPRESSION_DEPTHS_PATH + r"/Processed/"


# Create watershed subfolders in specified paths
SUBFOLDERS_LIST = [DD_RAW_PATH,
                   DD_INTERIM_PATH,
                   DD_PROCESSED_PATH]
for sub in SUBFOLDERS_LIST:
    os.makedirs(sub, exist_ok=True)

### 1.3 Move input data files to "Raw" subfolder

In [ ]:
# Specify source and destination folders before moving files
src = Path(BS_DATA_PATH)
dst = Path(DD_RAW_PATH)


# Move files in base data folder to watershed folder
for file in src.iterdir():
     if file.is_file():
        destination_file = dst / file.name
        if destination_file.exists():
            print(f"Skipped (already exists): {file.name}")
            continue
        shutil.move(str(file), destination_file)

## 2. Declare input data file names as variables

In [ ]:
# INPUTS
# Name of watershed shapefile from CLRH hydrofabrics zip file (.shp)
CLRH_FILENAME = usr_input().file("hydrofabric shapefile", "finalcat_info_v1-0")

# maybe provide input default value to user in the prompt

# Name of LiDAR DEM raster from LiDAR DEM zip file (.tif)
LIDAR_FILENAME = usr_input().file("LiDAR DEM raster", "sr_dem_cgvd28")


# Name of streams shapefile from NHN streams zip file (.shp)
NHN_FILENAME = usr_input().file("NHN streams shapefile", "NHN_05OE000_5_0_HD_NLFLOW_1")


## 3. Process LiDAR raster DEM

### 3.1 Project subbasins shapefile to align with LiDAR raster DEM

In [ ]:
# Load CLRH subbasins shapefile
clrh_gdf = gpd.read_file(DD_RAW_PATH + \
                        CLRH_FILENAME + \
                        ".shp")

In [ ]:
# Open LiDAR DEM and check CRS
with rasterio.open(DD_RAW_PATH + \
                   LIDAR_FILENAME + \
                   ".tif") as src:
    input_DEM_crs = src.crs


# Project subbasins data to match DEM
clrh_gdf_projected_lidar = clrh_gdf.to_crs(input_DEM_crs)


# Remove any non-alphanumeric characters from crs name
input_DEM_crs_alnum = ''.join(c for c in \
                              str(input_DEM_crs) if c.isalnum())


# Projected subbasins shapefile name with path
CLRH_PROJ_LIDAR_FILE = DD_INTERIM_PATH + \
                        CLRH_FILENAME + \
                        f"_projected_{input_DEM_crs_alnum}"


# Write projected subbasins data to shapefile
clrh_gdf_projected_lidar.to_file(CLRH_PROJ_LIDAR_FILE + ".shp")


# Check if shapefile projection aligns with DEM projection
is_correctly_projected_clrh_lidar = (input_DEM_crs == clrh_gdf_projected_lidar.crs)
                          

# Print results
print("Shapefile projection is aligned with DEM projection: ", 
      is_correctly_projected_clrh_lidar)
print("\nIf True, proceed to clipping.\n")

### 3.2 Clip LiDAR raster DEM to watershed

In [ ]:
# Convert projected subbasins data to GeoJSON-like format
shapes = [mapping(geom) for geom in clrh_gdf_projected_lidar.geometry]


# Mask (clip) the input DEM file
with rasterio.open(DD_RAW_PATH + \
                    LIDAR_FILENAME + \
                    ".tif") as src:
    nodata_value = src.nodata
    out_image, out_transform = mask(src, 
                                    shapes, 
                                    crop=True)
    out_meta = src.meta.copy()


# Update the copied metadata to match the clipped raster's dimensions and transform
out_meta.update({
    "driver": "GTiff",
    "height": out_image.shape[1],
    "width": out_image.shape[2],
    "transform": out_transform
})


# Clipped DEM file name with path
LIDAR_CLIPPED_FILE = DD_INTERIM_PATH + \
                        LIDAR_FILENAME + \
                        "_clip"


# Write clipped DEM data into file
with rasterio.open(LIDAR_CLIPPED_FILE + ".tif",
                   "w", 
                   **out_meta) as dest:
    dest.write(out_image)


# Print out some stats fyi
valid_out_image = out_image[0][out_image[0] != nodata_value]
print("Clipped unique values:", np.unique(valid_out_image))
print("Clipped number of unique values: ", len(np.unique(valid_out_image)))
print("Clipped maximum value :", np.max(valid_out_image))
print("Clipped minimum value (excluding nodata value): ",  np.min(valid_out_image))
print("\nIf the numbers make sense, proceed to resampling.\n")
""" #=======================ELENA CODE (WILL DELETE IT SOON)===================================
### SUBBASIN AND STREAM SHAPEFILES WERE PROJECTED IN ARCGIS PRO (NO PROJECTION TOOL IN WHITEBOX)

# Clip NHN streams to study area
# wbt.clip(
#    i = 'NHN_05OE000_5_0_HD_NLFLOW_1.shp',
#    clip = 'clrh_basins_projected_4617.shp',
#    output = 'NHN_Clip.shp'
#)

### RASTER DATA WAS CLIPPED AND RESAMPLED IN ARCGIS PRO DUE TO MEMORY ALLOCATION ERROR

# Clip dem raster to study area
#wbt.clip_raster_to_polygon(
    #i = 'sr_dem_cgvd28.tif',
    #polygons = 'finalcat_info_v1-0_Project.shp',
    #output = 'sr_dem_cgvd28_Clip_MC.tif'
#)

# Resample DEM to 5m (for faster processing)
#wbt.resample(
    #inputs = 'sr_dem_cgvd28_Clip_MC.tif',
    #output = 'sr_dem_cgvd28_Clip_MC_Resample_5m_corr',
    #cell_size = 5,
    #method = "cc"
#) """

In [ ]:
out_image[0]

### 3.3 Resample clipped LiDAR raster DEM to 5m resolution

In [ ]:
# Initialize whitebox tools object
wbt = WhiteboxTools()

In [ ]:
# Define resolution
resolution = 5 
resolution_units = "m" # metres


LIDAR_CLIPPED_RESAMPLED_FILE = LIDAR_CLIPPED_FILE + \
                                f"_resample_{resolution}{resolution_units}"
                                

# Resample to defined resolution
wbt.resample(
    inputs=LIDAR_CLIPPED_FILE+".tif",
    output=LIDAR_CLIPPED_RESAMPLED_FILE+".tif",
    cell_size=resolution,
    method="cc"
)


# Open the clipped resampled file to get some stats
with rasterio.open(LIDAR_CLIPPED_RESAMPLED_FILE + ".tif") as src:
    image = src.read(1) # Read first layer (only has one layer)


# Print out some stats fyi
valid_image = np.ma.masked_equal(image, nodata_value)
print("Clipped resampled DEM unique values:", np.unique(valid_image))
print("Clipped resampled number of unique values: ", len(np.unique(valid_image)))
print("Clipped resampled maximum value :", np.max(valid_image))
print("Clipped resampled minimum value (excluding nodata value): ",  np.min(valid_image))
print("\nIf the numbers make sense, proceed to next sections.")


## 4. Process NHN Streams Network Shapefile

### 4.1 Project subbasins shapefile to align with NHN streams shapefile

In [ ]:
# Load NHN streams shapefile
nhn_gdf = gpd.read_file(DD_RAW_PATH + \
                        NHN_FILENAME + \
                         ".shp")

In [ ]:
# CRS for NHN shapefile
input_NHN_crs = nhn_gdf.crs


# Project subbasins data to match streams data
clrh_gdf_projected_nhn = clrh_gdf.to_crs(input_NHN_crs)


# Remove any non-alphanumeric characters from crs name
input_NHN_crs_alnum = ''.join(c for c in \
                              str(input_NHN_crs) if c.isalnum())


# Projected subbasins shapefile name with path
CLRH_PROJ_NHN_FILE = DD_INTERIM_PATH + \
                        CLRH_FILENAME + \
                        f"_projected_{input_NHN_crs_alnum}" 


# Write projected subbasins data to shapefile
clrh_gdf_projected_nhn.to_file(CLRH_PROJ_NHN_FILE + \
                               ".shp")


# Check if shapefile projection aligns with DEM projection
is_correctly_projected_clrh_nhn = (input_NHN_crs == clrh_gdf_projected_nhn.crs)
                          

# Print results
print("Shapefile projection is aligned with NHN projection: ", 
      is_correctly_projected_clrh_nhn)
print(f"Projection is: {input_NHN_crs}")
print("\nIf True and EPSG:4617 are returned, proceed to clipping.")

### 4.2 Clip NHN streams shapefile to watershed

TS Note - Can we adjust this so the depressions raster and the other shapefiles (e.g. hydrofabric) have the same CRS at this point? 

In [ ]:
# Clipped NHN shapefule name with PATH
NHN_CLIPPED_FILE = DD_INTERIM_PATH + \
                  NHN_FILENAME + \
                  "_clip"

# Clip NHN streams shapefile to watershed
wbt.clip(
   i=DD_RAW_PATH+NHN_FILENAME+".shp",
   clip=CLRH_PROJ_NHN_FILE+".shp",
   output=NHN_CLIPPED_FILE+".shp"
)


# Load NHN streams shapefile
nhn_gdf_clip = gpd.read_file(NHN_CLIPPED_FILE + \
                             ".shp")


# Project clipped NHN shapefile to match DEM
nhn_gdf_clip_projected_lidar = nhn_gdf_clip.to_crs(input_DEM_crs)


# Clipped and projected NHN shapefile name with path
NHN_CLIPPED_PROJECTED_LIDAR_FILE = DD_INTERIM_PATH + \
                                    NHN_FILENAME + \
                                    f"_clip_projected_{input_DEM_crs_alnum}"


# Write clipped and projected NHN shapefile
nhn_gdf_clip_projected_lidar.to_file(NHN_CLIPPED_PROJECTED_LIDAR_FILE + \
                                     ".shp")


# Check if shapefile projection aligns with DEM projection
is_correctly_projected_nhn_lidar = (input_DEM_crs == nhn_gdf_clip_projected_lidar.crs)


# Print results
print("NHN shapefile projection is aligned with DEM projection: ", 
      is_correctly_projected_nhn_lidar)
print(f"Projection is: {input_DEM_crs}")
print("\nIf True and EPSG:26914 are returned, proceed to clipping.")

### 4.3 Visualize clipped and projected NHN streams shapefile

In [ ]:
# Visualize clipped and projected NHN streams shapefile
nhn_gdf_clip_projected_lidar.explore()

## 5. Generate depressions raster

In [ ]:
# Burn streams into DEM and fill
wbt.fill_burn(
    dem=LIDAR_CLIPPED_RESAMPLED_FILE+".tif",
    streams=NHN_CLIPPED_PROJECTED_LIDAR_FILE+".shp",
    output=LIDAR_CLIPPED_RESAMPLED_FILE+"_FillBurn"+".tif"
)


# Subtract raw DEM from filled DEM
first_statement_operand = LIDAR_CLIPPED_RESAMPLED_FILE+"_FillBurn"+".tif"
second_statement_operand = LIDAR_CLIPPED_RESAMPLED_FILE+".tif"
wbt.raster_calculator(
    output = LIDAR_CLIPPED_RESAMPLED_FILE+"_FillBurn_Deps"+".tif",
    statement = f"'{first_statement_operand}' - '{second_statement_operand}'"  
)


# Depressions raster file with path
DEPRESSIONS_RASTER_FILE = DD_PROCESSED_PATH+LIDAR_FILENAME+\
                        f"_clip_resample_{resolution}{resolution_units}_FillBurn_Deps_Corr"


# Remove stray burn lines
wbt.conditional_evaluation(
    i=LIDAR_CLIPPED_RESAMPLED_FILE+"_FillBurn_Deps"+".tif",
    output=DEPRESSIONS_RASTER_FILE+".tif",
    statement="value < 0.0",
    true=0.0,
    false=LIDAR_CLIPPED_RESAMPLED_FILE+"_FillBurn_Deps"+".tif"
)


## 6. Calculate Depression Depths

### 6.1 Calculate Subbasin Zonal Statistics

Note on zonal statistics generated with Whitebox Tools - [link to manual](https://www.whiteboxgeo.com/manual/wbt_book/available_tools/mathand_stats_tools.html#ZonalStatistics)

- It is unclear whether the calculation for "Total" accounts for the pixel resolution, or simply adds the values across all of the pixels. Based on a quick review of the raster and shapefile in ArcGIS, it seems to be the latter (pixel, not square meters). A factor of 25 needs to be added when calculating depression storage for Raven.

In [ ]:
# CLRH subbasins raster file name with path 
CLRH_RASTER_FILE = CLRH_PROJ_LIDAR_FILE+"_raster"


# Convert clrh subbasin polygons to raster
wbt.vector_polygons_to_raster(
    i=CLRH_PROJ_LIDAR_FILE+".shp",
    output=CLRH_RASTER_FILE+".tif",
    field = "FID",
    nodata = True,
    cell_size = 5.0
)


# Open clrh subbasins raster file
with rasterio.open(CLRH_RASTER_FILE+".tif") as src:
    target_crs = 26914
    profile = src.profile
    profile.update(crs=CRS.from_epsg(target_crs))


# Zonal stats file name with path
ZONAL_STATS_FILE = Path(DD_PROCESSED_PATH, 
                   f"ZonalStats_{WATERSHED_NAME}.html").resolve()


# Calculate zonal statistics
wbt.zonal_statistics(
    i=DEPRESSIONS_RASTER_FILE + ".tif",
    features=CLRH_RASTER_FILE + ".tif",
    stat="total",
    out_table=ZONAL_STATS_FILE
)

### 6.2 Calculate depression depths from zonal statistics

In [ ]:
# Read zonal stats html as pandas df
zonal_stats_df = pd.read_html(ZONAL_STATS_FILE,
                              flavor='bs4')[0]

# Calculate depression depths (mm) and add field to gdf
clrh_gdf_projected_lidar['Deps_Depth_mm'] = zonal_stats_df['Mean'] * 1000

# Calculate depression volumes (m3) and add field to gdf
clrh_gdf_projected_lidar['Deps_Vol_m3'] = clrh_gdf_projected_lidar['Deps_Depth_mm']*clrh_gdf_projected_lidar['BasArea']

# IY code below. TS simplified method above (removes need for Deps_SUM field in output dataset)

# # Join zonal stats totals to clrh subbasins geodataframe
# clrh_gdf_projected_lidar['Deps_SUM'] = zonal_stats_df['Total']


# # Calculate depression volumes (m3) and add field to gdf
# clrh_gdf_projected_lidar['Deps_Vol_m3'] = clrh_gdf_projected_lidar['Deps_SUM']*5*5


# # Calculate depression depths (mm) and add field to gdf
# clrh_gdf_projected_lidar['Deps_Depth_mm'] = clrh_gdf_projected_lidar['Deps_Vol_m3']/ \
#                                             clrh_gdf_projected_lidar['BasArea']*1000

### 6.3 Write depression depths to file

In [ ]:
# Depression depths file name with path
DEPRESSION_DEPTHS_FILE = DD_PROCESSED_PATH + \
                         "CLRH_basins_depression_depths"


# Write geodataframe to file for use in raven input file creation
clrh_gdf_projected_lidar.to_file(DEPRESSION_DEPTHS_FILE + ".shp")

## 7. Visualize Processed Data

In [ ]:
# Open the CLRH raster file
with rasterio.open(CLRH_RASTER_FILE + ".tif") as src:
    raster = src.read(1)
    nodata = src.nodata
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]


# Mask no-data values
raster = np.ma.masked_equal(raster, nodata)


# Open depression depths raster file
with rasterio.open(DEPRESSIONS_RASTER_FILE + ".tif") as src:
    depress = src.read(1)
    depress_nodata = src.nodata
    depress_extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]


# Mask no data values
depress = np.ma.masked_equal(depress, depress_nodata)


# Plot
fig, ax = plt.subplots(figsize=(10, 8))


# Plot the raster in grayscale
raster_img = ax.imshow(raster,
                    cmap='gray', 
                    extent=extent,
                    interpolation='nearest')


# Depression raster (overlaid in red or blue with transparency)
ax.imshow(depress, 
          cmap='Blues', 
          extent=depress_extent, 
          alpha=0.5) 


# Overlay the shapefile
nhn_gdf_clip_projected_lidar.plot(ax=ax, 
                                  facecolor='black', 
                                  edgecolor='black', 
                                  linewidth=1)


# Final touches
plt.title(f"{WATERSHED_NAME} Depression Depths")
plt.axis('off')
plt.show()

### OLD CODE, DO NOT RUN

In [ ]:
os.getcwd()

### RUNNING WHITEBOX TOOLS CHANGES THE WORKING DIRECTORY (ADDS WBT FOLDER). NOTEBOOK MUST BE RUN TOP TO BOTTOM, OTHERWISE PATHS WITLL NOT WORK

In [ ]:
# Project NHN clipped shapefile
streams_clip = gpd.read_file(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + "NHN_Clip.shp")
streams_clip_proj_26914 = streams_clip.to_crs(26914)
streams_clip_proj_26914.to_file(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + "NHN_Clip_Project_26914.shp")

## Generate Depressions Raster

In [ ]:
# Burn streams into DEM and fill
wbt.fill_burn(
    dem = 'sr_dem_cgvd28_Clip_MC_Resample_5m_corr.tif',
    streams = 'NHN_Clip_Project_26914.shp',
    output = 'DEM_Clip_MC_Resample_5m_FillBurn.tif'
)

# Subtract raw DEM from filled DEM
wbt.raster_calculator(
    output = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps.tif',
    statement = "'DEM_Clip_MC_Resample_5m_FillBurn.tif' - 'sr_dem_cgvd28_Clip_MC_Resample_5m_corr.tif'"
)

# Remove stray burn lines
wbt.conditional_evaluation(
    i = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps.tif',
    output = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps_Corr.tif',
    statement = "value < 0.0",
    true = 0.0,
    false = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps.tif'
)

In [ ]:
# Convert clrh subbasin polygons to raster
wbt.vector_polygons_to_raster(
    i = 'clrh_basins_projected_26914.shp',
    output = 'clrh_basins_raster.tif',
    field = "FID",
    nodata = True,
    cell_size = 5.0
)

# Define projection of clrh raster
with rasterio.open(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + 'clrh_basins_raster.tif', 'r+') as rds:
    rds.crs = CRS.from_epsg(26914)

# Calculate zonal statistics
wbt.zonal_statistics(
    i = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps_Corr.tif',
    features = 'clrh_basins_raster.tif',
    stat="total",
    out_table='ZonalStats_DEM_Clip_MC_Resample_5m_FillBurn_Deps_Corr.html'
)

In [ ]:
# Read html file as pandas dataframe
file = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + 'ZonalStats_DEM_Clip_MC_Resample_5m_FillBurn_Deps_Corr.html'
zonal_stats_dfs = pd.read_html(file, flavor='bs4')
zonal_stats_df = zonal_stats_dfs[0]

# Join totals to clrh subbasins geodataframe
clrh_gdf_projected_26914['Deps_SUM'] = zonal_stats_df['Total']

# Calculate depression volumes (m3) and add field to gdf
clrh_gdf_projected_26914['Deps_Vol_m3'] = clrh_gdf_projected_26914['Deps_SUM'] * 5 * 5

# Calculate depression depths (mm) and add field to gdf
clrh_gdf_projected_26914['Deps_Depth_mm'] = clrh_gdf_projected_26914['Deps_Vol_m3'] / clrh_gdf_projected_26914['BasArea'] * 1000

In [ ]:
# Write geodataframe to file for use in raven input file creation
clrh_gdf_projected_26914.to_file(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + "clrh_basins_deps.shp")
